In [46]:
# Import packages
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go

# Set options
pd.set_option('display.max_columns', 55)

# Read pre processed parquet
clean_data = pd.read_parquet("sample_by_gene.parquet")

In [47]:
# Standardize and run PCA
X = clean_data.drop(columns=['Target(SNHG14)', 'sample_type']).values

pca = PCA(n_components=2)
pcs = pca.fit_transform(StandardScaler().fit_transform(X))


In [48]:
fig2 = px.scatter(
    pca_df,
    x='PC1', y='PC2',
    color='sample_type',
    text='sample_type',
    hover_name=pca_df.index,
    labels={
        'PC1': f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
        'PC2': f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
    },
    title='PCA of Gene Expression by All Sample Types (Safety)'
          '<br><sup>Treatments closest to Control are safest</sup>',
)

fig2.update_traces(mode='text')
fig2.update_layout(showlegend=False)

for trace in fig2.data:
    if trace.name == 'Control':
        trace.textfont.color = 'black'
        trace.textfont.size = 14
        trace.textfont.weight = 'bold'
    else:
        trace.textfont.color = trace.marker.color
        trace.textfont.size = 10

# Add Control centroid as a distinct labeled point
control_centroid = pca_df[pca_df['sample_type'] == 'Control'][['PC1', 'PC2']].mean()
fig2.add_trace(go.Scatter(
    x=[control_centroid['PC1']],
    y=[control_centroid['PC2']],
    mode='text',
    text=['⊕Mean'],
    textfont=dict(color='black', size=16, weight='bold'),
    hoverinfo='skip',
    showlegend=False,
))

fig2.show()


Both hATF555R replicates are closest to the mean between the two control points. 

In [49]:
# Mean Target(SNHG14) expression per sample type
target_mean = clean_data.groupby('sample_type')['Target(SNHG14)'].mean()

# Log2 fold change relative to Control mean (-1 -> 50% reduction, -2 -> 75% reduction, etc.)
control_mean = target_mean['Control']
log2fc = np.log2(target_mean / control_mean).drop('Control').sort_values()

fig3 = px.bar(
    log2fc,
    x=log2fc.index,
    y=log2fc.values,
    labels={'x': 'Treatment', 'y': 'Log2 Fold Change vs Control'},
    title='Target Gene (SNHG14) Log2 Fold Change by Treatment (Effectiveness)'
          '<br><sup>Log2FC: −1 = halved (50% reduction), −2 = quartered (75% reduction), 0 = no change relative to control</sup>',
    color=log2fc.values,
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
)
fig3.add_hline(y=0, line_dash='dash', line_color='black')
fig3.update_layout(coloraxis_showscale=False, xaxis_tickangle=-45)

# Right axis: % change at specific log2FC reference points
tick_log2fc = np.array([-1.2, -1.0, -0.8, -0.6, -0.4, -0.2, 0.0])
tick_pct = (2 ** tick_log2fc - 1) * 100

fig3.add_trace(go.Scatter(x=[None], y=[None], yaxis='y2', showlegend=False))

y_min = log2fc.min() * 1.15
y_max = log2fc.max() * 1.15

fig3.update_layout(
    yaxis=dict(range=[y_min, y_max]),
    yaxis2=dict(
        overlaying='y',
        side='right',
        range=[y_min, y_max],
        tickmode='array',
        tickvals=tick_log2fc,
        ticktext=[f'{p:.0f}%' for p in tick_pct],
        title='% Change vs Control',
    ),
)

fig3.show()

# # Summary table sorted most-to-least effective (most negative log2FC first)
# summary = pd.DataFrame({
#     'mean_CPM': target_mean.drop('Control'),
#     'control_mean_CPM': control_mean,
#     'log2FC': log2fc,
#     'pct_change': ((target_mean.drop('Control') - control_mean) / control_mean * 100).round(1),
# }).sort_values('log2FC')
# summary


The most effective treatment is nZF139 which more than halves the expression of the target gene on average

In [50]:
# Filter to Control + top 10 most effective treatments
top10_types = list(log2fc.index[:10]) + ['Control']
pca_df_top10 = pca_df[pca_df['sample_type'].isin(top10_types)].copy()

# Build legend labels: "SampleType (log2FC: -1.23)" for treatments, "Control" for control
label_map = {t: f'{t} (log2FC: {log2fc[t]:.2f})' for t in log2fc.index[:10]}
label_map['Control'] = 'Control'
pca_df_top10['label'] = pca_df_top10['sample_type'].map(label_map)

# Legend order: most negative log2FC first, Control last
legend_order = [label_map[t] for t in log2fc.index[:10]] + ['Control']

fig4 = px.scatter(
    pca_df_top10,
    x='PC1', y='PC2',
    color='label',
    hover_name=pca_df_top10.index,
    category_orders={'label': legend_order},
    color_discrete_sequence=px.colors.qualitative.T10,
    labels={
        'PC1': f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
        'PC2': f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
        'label': 'Sample Type',
    },
    title='PCA of Gene Expression — Top 10 Most Effective Treatments',
)

fig4.for_each_trace(
    lambda t: t.update(marker_size=18, marker_symbol='star', marker_line_width=2, marker_line_color='black')
    if t.name == 'Control' else t.update(marker_size=12)
)

# Add Control centroid
control_centroid = pca_df[pca_df['sample_type'] == 'Control'][['PC1', 'PC2']].mean()
fig4.add_trace(go.Scatter(
    x=[control_centroid['PC1']],
    y=[control_centroid['PC2']],
    mode='markers+text',
    marker=dict(symbol='cross', size=20, color='black', line=dict(width=2, color='black')),
    text=['Control Mean'],
    textposition='top center',
    textfont=dict(color='black', size=12, weight='bold'),
    hoverinfo='skip',
    showlegend=False,
))

fig4.show()


Using the average of both controls, nZFD96 is likely consistently closest with a log2FC of -0.45 implying the treatment decreased  by roughly 27%.


In [51]:
# Euclidean distance of each sample to the Control centroid in PCA space
control_centroid = pca_df[pca_df['sample_type'] == 'Control'][['PC1', 'PC2']].mean()

pca_df['dist_to_control'] = np.sqrt(
    (pca_df['PC1'] - control_centroid['PC1'])**2 +
    (pca_df['PC2'] - control_centroid['PC2'])**2
)

# Mean ± std distance per sample type, sorted closest first (excluding Control)
dist_summary = (
    pca_df[pca_df['sample_type'] != 'Control']
    .groupby('sample_type')['dist_to_control']
    .agg(mean_dist='mean', std_dist='std', n='count')
    .sort_values('mean_dist')
)
dist_summary['log2FC'] = log2fc.reindex(dist_summary.index)

fig5 = px.bar(
    dist_summary,
    x=dist_summary.index,
    y='mean_dist',
    error_y='std_dist',
    color='log2FC',
    color_continuous_scale='Reds_r',
    range_color=[log2fc.min(), 0],
    labels={'x': 'Treatment', 'mean_dist': 'Mean Distance to Control Centroid (PCA)', 'log2FC': 'Log2FC'},
    title='Consistency with Control by Sample Type — Euclidean Distance in PCA Space'
          '<br><sup>Bar height = PCA distance to Control (lower = safer); color = log2FC effectiveness (darker red = more effective)</sup>',
)
fig5.update_layout(xaxis_tickangle=-45)
fig5.show()

# dist_summary


In [52]:
import plotly.express as px  # rebind in case overwritten by prior run

# Effectiveness vs Safety tradeoff scatter
# x = log2FC (effectiveness), y = mean distance to Control (safety)
# bubble size = std_dist (replicate consistency — smaller = more consistent)

tradeoff_df = dist_summary.copy()

# Pareto frontier: non-dominated set minimizing both log2FC and mean_dist
sorted_df = tradeoff_df.sort_values('log2FC')
pareto = []
min_dist = float('inf')
for name, row in sorted_df.iterrows():
    if row['mean_dist'] < min_dist:
        pareto.append((row['log2FC'], row['mean_dist'], name))
        min_dist = row['mean_dist']
pareto_x = [p[0] for p in pareto]
pareto_y = [p[1] for p in pareto]

step_x, step_y = [], []
for i, (fx, fy) in enumerate(zip(pareto_x, pareto_y)):
    if i == 0:
        step_x += [tradeoff_df['log2FC'].min() * 1.05, fx]
        step_y += [fy, fy]
    else:
        step_x += [pareto_x[i - 1], fx]
        step_y += [fy, fy]
step_x.append(fx)
step_y.append(0)

med_log2fc = tradeoff_df['log2FC'].median()
med_dist = tradeoff_df['mean_dist'].median()

fig6 = px.scatter(
    tradeoff_df,
    x='log2FC',
    y='mean_dist',
    size='std_dist',
    size_max=25,
    text=tradeoff_df.index,
    color=tradeoff_df.index,
    color_discrete_sequence=px.colors.qualitative.Dark24,
    labels={
        'log2FC': 'Log2 Fold Change (lower = more effective)',
        'mean_dist': 'Mean Distance to Control Centroid (lower = safer)',
    },
    title='Effectiveness vs Safety Tradeoff by Treatment'
          '<br><sup>Bubble size = replicate std dev (larger = less consistent); Pareto frontier = optimal tradeoff candidates</sup>',
)
fig6.update_traces(
    mode='markers+text',
    textposition='middle center',
    textfont=dict(color='black', size=11),
)

fig6.add_trace(go.Scatter(
    x=step_x, y=step_y,
    mode='lines',
    line=dict(color='black', width=2, dash='dot'),
    name='Pareto Frontier',
    showlegend=True,
))

fig6.add_vline(x=med_log2fc, line_dash='dash', line_color='gray', opacity=0.5)
fig6.add_hline(y=med_dist, line_dash='dash', line_color='gray', opacity=0.5)

# Corner labels using paper coordinates (0–1) to anchor at plot edges
for x, y, text, color, xanchor, yanchor in [
    (0.01, 0.01, '✓ Safe & Effective',  'green',     'left',  'bottom'),
    (0.99, 0.01, 'Safe but Weak',        'steelblue', 'right', 'bottom'),
    (0.01, 0.99, 'Effective but Risky',  'orange',    'left',  'top'),
    (0.99, 0.99, '✗ Unsafe & Weak',      'red',       'right', 'top'),
]:
    fig6.add_annotation(
        x=x, y=y, text=text,
        xref='paper', yref='paper',
        showarrow=False,
        font=dict(color=color, size=11),
        xanchor=xanchor, yanchor=yanchor,
    )

fig6.update_layout(showlegend=False)
fig6.show()


**Interpreting the Tradeoff Plot**

Each bubble is a treatment. Its position encodes two metrics: how far left it sits reflects effectiveness (more negative log2FC = greater SNHG14 target gene reduction), and how low it sits reflects safety (shorter PCA distance to Control = less disruption/variation to all other genes). Bubble size encodes replicate consistency — smaller bubbles indicate more reproducible results across replicates.

The dotted Pareto frontier marks treatments where no single alternative is better on *both* axes at once. Moving left along the frontier gains effectiveness but costs safety, and vice versa. Treatments off the frontier are dominated — there exists at least one other treatment that is simultaneously more effective and safer. Theoretically, we should be indifferent between any of the treatments on the pareto frontier, and never chose a treatment not on the frontier. 

The ideal candidate sits as far into the bottom-left quadrant as possible with a small bubble: maximally effective, minimally disruptive, and highly consistent.

The 9 pareto optimal treatments include: nZF139, nZF153, nZFD96, nZF151, nZF81, Base, hATF555Q, nZF36, and hATF555R. 

nZF139 is the most effective but least safe of these while hATF555R is the safest but least effective. One can choose their ideal treatment type based on their desired level of effectivness and treatment.